In [1]:

import re
from google.genai import types
import sys
sys.path.append("D:\\piyush_work\\agentic_orchestrator_adk")
from SharedResources.load_environment import get_client
from chromadb import PersistentClient
CHROMA_COLLECTION = "incident_embeddingsv2_ss"
CHROMA_PERSIST_PATH = "chromadb"

######## NEW DATA FETCHING IMPLEMENTATION #######
#################################################


chroma_client = PersistentClient(path=CHROMA_PERSIST_PATH)
client = get_client()

# --- helpers for key compatibility and cleaning ---

def _clean(s: str) -> str:
    if s is None:
        return ""
    s = str(s).replace("\u0000", " ")
    return re.sub(r"\s+", " ", s).strip()

def _get(row: dict, *candidates: str) -> str:
    """Try multiple key variants and return the first non-empty value."""
    for k in candidates:
        v = row.get(k)
        if v is not None and str(v).strip() != "":
            return str(v).strip()
    return ""


# --- embeddings ---
#while creating incidents
def get_embeddings(text: str, model_id: str):
    """
    Generate embedding from already prepared text.
    """
    text = _clean(text)

    response = client.models.embed_content(
        model=model_id,
        contents=text,
        config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY",
                                        # output_dimensionality=768
                                        ),
    )

    emb = response.embeddings[0]
    values = getattr(emb, "values", emb)
    return list(values)


environment varialble loaded!
Client created successfully!


In [ ]:
import csv
from datetime import datetime

# --- ingestion ---

def add_document_from_file(file_path: str, collection_name: str, model_id: str = "text-embedding-005"):
    """
    Load CSV file and add documents to ChromaDB collection with rich metadata.
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)

            collection = chroma_client.get_or_create_collection(name=collection_name)

            count = 0
            for incident_data in reader:      
                # --- Extract fields safely ---
                doc_id = _get(incident_data, "Incident ID", "Incident_id")
                if not doc_id:
                    print("[WARN] Skipping row without Incident ID")
                    continue

                title = _get(incident_data, "Incident Title", "Incident_title")
                description = _get(incident_data, "Incident Description", "Incident_description")
                cause = _get(incident_data, "Root Cause", "Incident_cause")
                resolution = _get(incident_data, "Resolution", "Incident_resolution")

                app_id = _get(incident_data, "Application ID", "Application_id")
                severity = _get(incident_data, "Severity")

                created_ts = _get(incident_data, "Incident Creation Timestamp")
                resolved_ts = _get(incident_data, "Resolution Timestamp")

                # --- Optional: derive useful features ---
                resolution_time_mins = None
                try:
                    if created_ts and resolved_ts:
                        fmt = "%Y-%m-%d %H:%M:%S"
                        delta = datetime.strptime(resolved_ts, fmt) - datetime.strptime(created_ts, fmt)
                        resolution_time_mins = int(delta.total_seconds() / 60)
                except Exception:
                    pass  # don't break ingestion for bad timestamps

                # --- Build embedding text (THIS is critical) ---
                embedding_text = f"""
                Incident Title: {title}
                Incident Description: {description}
                """

                embedding = get_embeddings(embedding_text, model_id)

                # --- Add to collection ---
                collection.add(
                    ids=[doc_id],
                    documents=[embedding_text],
                    embeddings=[embedding],
                    metadatas=[{
                        "incident_id": doc_id,
                        "application_id": app_id,
                        "title": title,
                        "description": description,
                        "root_cause": cause,
                        "resolution": resolution,
                        "severity": severity,
                        "created_ts": created_ts,
                        "resolved_ts": resolved_ts,
                        "resolution_time_mins": resolution_time_mins
                    }]
                )

                count += 1
                print(f"Added document {count}: {doc_id}")

            print(f"\nSuccessfully added {count} documents to collection '{collection_name}'")
            return f"Success: {count} documents added"

    except FileNotFoundError:
        return f"Error: File not found at {file_path}"
    # except Exception as e:
    #     return f"There was an error loading the documents.\n{e}"

In [5]:
with open("incident_data.csv", 'r', encoding='utf-8') as file:
    reader = csv.DictReader(file)

In [3]:
add_document_from_file(file_path="incident_data.csv",collection_name=CHROMA_COLLECTION)

<class 'dict'>
Added document 1: INC3001
<class 'dict'>
Added document 2: INC3002
<class 'dict'>
Added document 3: INC3003
<class 'dict'>
Added document 4: INC3004
<class 'dict'>
Added document 5: INC3005
<class 'dict'>
Added document 6: INC3006
<class 'dict'>
Added document 7: INC3007
<class 'dict'>
Added document 8: INC3008
<class 'dict'>
Added document 9: INC3009
<class 'dict'>
Added document 10: INC3010
<class 'dict'>
Added document 11: INC3011
<class 'dict'>
Added document 12: INC3012
<class 'dict'>
Added document 13: INC3013
<class 'dict'>
Added document 14: INC3014
<class 'dict'>
Added document 15: INC3015
<class 'dict'>
Added document 16: INC3016
<class 'dict'>
Added document 17: INC3017
<class 'dict'>
Added document 18: INC3018
<class 'dict'>
Added document 19: INC3019
<class 'dict'>
Added document 20: INC3020
<class 'dict'>
Added document 21: INC3021
<class 'dict'>
Added document 22: INC3022
<class 'dict'>
Added document 23: INC3023
<class 'dict'>
Added document 24: INC3024
<

'Success: 100 documents added'